In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../assets/json")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:

import altair as alt
import pandas as pd

# allow large datasets for interactive filtering
alt.data_transformers.disable_max_rows()

# Load subsets
pisa_cols = [
    "CNT", "ESCS", "MATHEFF", "MATHPERS", "ST004D01T",
    "W_FSTUWT"
]
hsls_cols = [
    "X1SES", "X1MTHEFF", "X1STU30OCC_STEM1", "X1SEX"
]
pisa_df = pd.read_csv(DATA_DIR / "pisa_subset.csv", usecols=pisa_cols)
hsls_df = pd.read_csv(DATA_DIR / "hsls_subset.csv", usecols=hsls_cols)

# Continent/region mapping (fallback to "Other")
continent_map = {
    # North America / Latin
    "USA": "North America", "CAN": "North America", "MEX": "North America", "PAN": "North America", "CRI": "North America", "DOM": "North America",
    "JAM": "North America", "PRI": "North America", "BRA": "South America", "ARG": "South America", "CHL": "South America", "COL": "South America",
    "PER": "South America", "URY": "South America", "ECU": "South America",
    # Europe
    "GBR": "Europe", "FRA": "Europe", "DEU": "Europe", "ESP": "Europe", "ITA": "Europe", "PRT": "Europe", "NLD": "Europe", "BEL": "Europe",
    "LUX": "Europe", "CHE": "Europe", "AUT": "Europe", "SWE": "Europe", "NOR": "Europe", "DNK": "Europe", "FIN": "Europe", "ISL": "Europe",
    "IRL": "Europe", "POL": "Europe", "CZE": "Europe", "SVK": "Europe", "HUN": "Europe", "SVN": "Europe", "EST": "Europe", "LVA": "Europe",
    "LTU": "Europe", "GRC": "Europe", "TUR": "Europe", "ROU": "Europe", "BGR": "Europe", "HRV": "Europe", "SRB": "Europe", "MNE": "Europe",
    "ALB": "Europe", "BIH": "Europe", "MKD": "Europe",
    # Asia / Middle East
    "CHN": "Asia", "HKG": "Asia", "MAC": "Asia", "TWN": "Asia", "JPN": "Asia", "KOR": "Asia", "SGP": "Asia", "THA": "Asia",
    "VNM": "Asia", "IDN": "Asia", "MYS": "Asia", "PHL": "Asia", "KAZ": "Asia", "QAT": "Asia", "ARE": "Asia", "SAU": "Asia",
    "JOR": "Asia", "LBN": "Asia", "KWT": "Asia", "OMN": "Asia", "BHR": "Asia", "IND": "Asia", "PAK": "Asia", "BGD": "Asia",
    "LAO": "Asia", "KHM": "Asia",
    # Africa
    "ZAF": "Africa", "MAR": "Africa", "TUN": "Africa", "EGY": "Africa", "SEN": "Africa",
    # Oceania
    "AUS": "Oceania", "NZL": "Oceania", "FJI": "Oceania",
}

# Add continent labels and source
pisa_df = pisa_df.assign(source="PISA")
pisa_df["continent"] = pisa_df["CNT"].map(continent_map).fillna("Other")
hsls_df = hsls_df.assign(CNT="USA", continent="North America", source="HSLS")

# Harmonize and standardize within source before continent aggregation
base = pd.concat([
    pisa_df.rename(columns={"ESCS": "escs", "MATHEFF": "matheff"})[["continent", "escs", "matheff", "source"]],
    hsls_df.rename(columns={"X1SES": "escs", "X1MTHEFF": "matheff"})[["continent", "escs", "matheff", "source"]],
], ignore_index=True)
base = base.dropna(subset=["escs", "matheff", "continent", "source"])
base["z_escs"] = base.groupby("source")["escs"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
base["z_matheff"] = base.groupby("source")["matheff"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
continent_df = (
    base.groupby("continent")
    .agg(avg_escs=("z_escs", "mean"), avg_matheff=("z_matheff", "mean"), n=("z_escs", "size"))
    .reset_index()
)

# Student-level data for right chart (with continent) and combined source label
pisa_students = (
    pisa_df.rename(columns={"ST004D01T": "gender"})
    .assign(
        continent=lambda d: d["CNT"].map(continent_map).fillna("Other"),
        source="PISA",
        gender=lambda d: d["gender"].map({1: "Female", 2: "Male"}),
        stem_interest=lambda d: d["MATHPERS"],
    )
    [["continent", "gender", "stem_interest", "source"]]
    .dropna(subset=["stem_interest", "gender", "continent"])
)
hsls_students = (
    hsls_df.rename(columns={"X1SEX": "gender"})
    .assign(
        source="HSLS",
        gender=lambda d: d["gender"].map({1: "Male", 2: "Female"}),
        stem_interest=lambda d: d["X1STU30OCC_STEM1"],
    )
    [["continent", "gender", "stem_interest", "source"]]
    .dropna(subset=["stem_interest", "gender", "continent"])
)
students_df = pd.concat([pisa_students, hsls_students], ignore_index=True)

# Selection by continent
continent_select = alt.selection_point(fields=["continent"], name="continent_select")

left = (
    alt.Chart(continent_df)
    .mark_circle()
    .encode(
        x=alt.X("avg_escs:Q", title="Mean SES (z within source)"),
        y=alt.Y("avg_matheff:Q", title="Mean Math Self-Efficacy (z within source)"),
        size=alt.Size("n:Q", title="Sample Size", scale=alt.Scale(range=[60, 500])),
        color=alt.Color("continent:N", title="Continent/Region"),
        opacity=alt.condition(continent_select, alt.value(1), alt.value(0.5)),
        tooltip=["continent", "avg_escs", "avg_matheff", "n"],
    )
    .add_params(continent_select)
    .properties(title="Continent SES vs Math Self-Efficacy (Standardized)")
)

# Right chart: density plot by gender (aggregated for performance)
right = (
    alt.Chart(students_df)
    .transform_filter(continent_select)
    .transform_density(
        density="stem_interest",
        groupby=["gender"],
        as_=["stem_interest", "density"],
    )
    .mark_area(opacity=0.5)
    .encode(
        x=alt.X("stem_interest:Q", title="STEM Interest Index"),
        y=alt.Y("density:Q", title="Density"),
        color=alt.Color("gender:N", title="Gender"),
    )
    .properties(title="STEM Interest by Gender (Selected Continent)")
)
(left | right).resolve_scale(color="independent")


In [ ]:
import altair as alt
import pandas as pd
import numpy as np

alt.data_transformers.disable_max_rows()

# Columns
pisa_cols = ["CNT", "ICTRES", "MATHEFF", "MATHPERS", "PV1MATH", "ST004D01T", "IC170Q01JA", "IC170Q02JA"]
hsls_cols = ["X1SES", "X1MTHEFF", "X1TXMTSCOR", "X1SEX", "X1MTHINT", "S1WEBINFO"]

pisa_header = pd.read_csv(DATA_DIR / "pisa_subset.csv", nrows=0).columns
pisa_use = [c for c in pisa_cols if c in pisa_header]
pisa_df = pd.read_csv(DATA_DIR / "pisa_subset.csv", usecols=pisa_use)
hsls_df = pd.read_csv(DATA_DIR / "hsls_subset.csv", usecols=[c for c in hsls_cols if c in pd.read_csv(DATA_DIR/"hsls_subset.csv", nrows=0).columns])

continent_map = {
    "USA": "North America", "CAN": "North America", "MEX": "North America", "PAN": "North America", "CRI": "North America", "DOM": "North America",
    "JAM": "North America", "PRI": "North America", "BRA": "South America", "ARG": "South America", "CHL": "South America", "COL": "South America",
    "PER": "South America", "URY": "South America", "ECU": "South America",
    "GBR": "Europe", "FRA": "Europe", "DEU": "Europe", "ESP": "Europe", "ITA": "Europe", "PRT": "Europe", "NLD": "Europe", "BEL": "Europe",
    "LUX": "Europe", "CHE": "Europe", "AUT": "Europe", "SWE": "Europe", "NOR": "Europe", "DNK": "Europe", "FIN": "Europe", "ISL": "Europe",
    "IRL": "Europe", "POL": "Europe", "CZE": "Europe", "SVK": "Europe", "HUN": "Europe", "SVN": "Europe", "EST": "Europe", "LVA": "Europe",
    "LTU": "Europe", "GRC": "Europe", "TUR": "Europe", "ROU": "Europe", "BGR": "Europe", "HRV": "Europe", "SRB": "Europe", "MNE": "Europe",
    "ALB": "Europe", "BIH": "Europe", "MKD": "Europe",
    "CHN": "Asia", "HKG": "Asia", "MAC": "Asia", "TWN": "Asia", "JPN": "Asia", "KOR": "Asia", "SGP": "Asia", "THA": "Asia",
    "VNM": "Asia", "IDN": "Asia", "MYS": "Asia", "PHL": "Asia", "KAZ": "Asia", "QAT": "Asia", "ARE": "Asia", "SAU": "Asia",
    "JOR": "Asia", "LBN": "Asia", "KWT": "Asia", "OMN": "Asia", "BHR": "Asia", "IND": "Asia", "PAK": "Asia", "BGD": "Asia",
    "LAO": "Asia", "KHM": "Asia",
    "ZAF": "Africa", "MAR": "Africa", "TUN": "Africa", "EGY": "Africa", "SEN": "Africa",
    "AUS": "Oceania", "NZL": "Oceania", "FJI": "Oceania"
}

# STEM interest: use PISA MATHPERS, HSLS X1MTHINT
pisa_df = pisa_df.assign(stem_interest=pisa_df.get("MATHPERS"))
# ICT behavior: PISA use IC170Q01JA/02JA average if both present
if "IC170Q01JA" in pisa_df.columns and "IC170Q02JA" in pisa_df.columns:
    pisa_df["ict_behavior"] = pisa_df[["IC170Q01JA", "IC170Q02JA"]].mean(axis=1)
elif "IC170Q01JA" in pisa_df.columns:
    pisa_df["ict_behavior"] = pisa_df["IC170Q01JA"]
else:
    pisa_df["ict_behavior"] = np.nan

pisa_df = pisa_df.assign(
    source="PISA",
    continent=lambda d: d["CNT"].map(continent_map).fillna("Other"),
    gender=lambda d: d.get("ST004D01T", pd.Series(index=d.index)).map({1: "Female", 2: "Male"}),
    ict_resource=lambda d: d.get("ICTRES", pd.Series(index=d.index)),
)

hsls_df = hsls_df.assign(
    CNT="USA",
    source="HSLS",
    continent="North America",
    gender=lambda d: d["X1SEX"].map({1: "Male", 2: "Female"}),
    stem_interest=lambda d: d["X1MTHINT"],
    ict_resource=lambda d: d["X1SES"],
    ict_behavior=lambda d: d.get("S1WEBINFO", pd.Series(index=d.index)),
)

# Standardize resource and interest within source for left chart
left_base = pd.concat([
    pisa_df[["continent", "ict_resource", "stem_interest", "source"]],
    hsls_df[["continent", "ict_resource", "stem_interest", "source"]],
], ignore_index=True)
left_base = left_base.dropna(subset=["ict_resource", "stem_interest", "continent", "source"])
left_base["z_resource"] = left_base.groupby("source")["ict_resource"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
left_base["z_stem"] = left_base.groupby("source")["stem_interest"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
continent_df = (
    left_base.groupby("continent")
    .agg(avg_res=("z_resource", "mean"), avg_stem=("z_stem", "mean"), n=("z_resource", "size"))
    .reset_index()
)

# Student-level data with cleaning
pisa_students = pisa_df[["continent", "gender", "ict_behavior", "stem_interest", "source"]].dropna()
hsls_students = hsls_df[["continent", "gender", "ict_behavior", "stem_interest", "source"]].dropna()
students_df = pd.concat([pisa_students, hsls_students], ignore_index=True)
students_df = students_df[~students_df["ict_behavior"].isin([-9, -8, -7, -5])]

continent_select3 = alt.selection_point(fields=["continent"], name="continent_select3")

left = (
    alt.Chart(continent_df)
    .mark_circle()
    .encode(
        x=alt.X("avg_res:Q", title="Mean ICT Resources (z within source)"),
        y=alt.Y("avg_stem:Q", title="Mean STEM Interest (z within source)"),
        size=alt.Size("n:Q", title="Sample Size", scale=alt.Scale(range=[60, 500])),
        color=alt.Color("continent:N", title="Continent/Region"),
        opacity=alt.condition(continent_select3, alt.value(1), alt.value(0.5)),
        tooltip=["continent", "avg_res", "avg_stem", "n"],
    )
    .add_params(continent_select3)
    .properties(title="STEM Interest vs ICT Resources (Continent, Standardized)")
)

# Right chart: KDE by gender with bandwidth smoothing
right = (
    alt.Chart(students_df)
    .transform_filter(continent_select3)
    .transform_density(
        density="ict_behavior",
        groupby=["gender"],
        as_=["ict_behavior", "density"],
        bandwidth=0.25,
    )
    .mark_area(opacity=0.5)
    .encode(
        x=alt.X("ict_behavior:Q", title="ICT Behavior (proxy)"),
        y=alt.Y("density:Q", title="Density"),
        color=alt.Color("gender:N", title="Gender"),
        tooltip=["gender:N", "ict_behavior:Q", "density:Q"],
    )
    .properties(title="ICT Behavior Distribution by Gender (Selected Continent)")
)

(left | right).resolve_scale(color="independent")
